#### Reverse-Engineering Binary WASM

In [ ]:
def runpywasm(wasmfile):
  from pywasm.core import Machine, Runtime, FuncType, ValType

  # P0lib implementation in Python
  def write(_: Machine, args: list[int]) -> list[int]:
    print(args[0])
    return []

  def writeln(_: Machine, args: list[int]) -> list[int]:
    print()
    return []

  def read(_: Machine, args: list[int]) -> list[int]:
    return [int(input())]

  # Create runtime
  runtime = Runtime()
  runtime.imports = {
    'P0lib': {
      'write': runtime.allocate_func_host(FuncType([ValType.i32()], []), write),
      'writeln': runtime.allocate_func_host(FuncType([], []), writeln),
      'read': runtime.allocate_func_host(FuncType([], [ValType.i32()]), read),
    }
  }
  # Create and run instance
  instance = runtime.instance_from_file(wasmfile)

In [ ]:
def hexdump(fn: str):
  with open(fn, 'rb') as hexfile:  # open binary file for reading
    word, pos = hexfile.read(4), 0  # type(word) == bytes
    while len(word) > 0:
      print(
        '{:#08x}'.format(pos)
        + ': '
        + ' '.join('{:02X}'.format(b) for b in word)
        + '    '
        + ''.join(chr(c) if 32 <= c < 127 else '.' for c in word)
      )
      word, pos = hexfile.read(4), pos + 4

In [ ]:
import import_ipynb
from P0 import compileString

Consider the following P0 program. Compile it first to a textual .wat file and then convert it to a binary file:

In [ ]:
compileString(
  """
const N = 5
var a: [1 .. N] → integer
program seven
  var i : integer
    a[1] := 3; a[2] := 7; a[3] := 5; a[4] := 9; a[5] := 7
    i := 1
    while i ≤ N do
      if a[i] = 7 then write(i)
      i := i + 1
""",
  'seven.wat',
)

In [ ]:
!wat2wasm seven.wat

You may check that it works as intended by executing the binary file:

In [ ]:
runpywasm('seven.wasm')

In [ ]:
!cat seven.wat

In [ ]:
!wasm2wat seven.wasm

Now "reverse engineer" `seven.wasm`:

In [ ]:
hexdump('seven.wasm')

1. How long is the generated code in (decimal) bytes?

YOUR ANSWER HERE

2. Annotate the hex dump of `seven.wasm` in the same way as `max.wasm` In the WASM Binary Format Tutorial!

YOUR ANSWER HERE